In [ ]:
# Pareto-style cumulative distribution of output tokens + class composition
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.patches import Rectangle

dataset_path = "dataset/final_binary/final_dataset.csv"
dataset_df = pd.read_csv(dataset_path)

required_cols = {"id", "output_tokens_gpt5", "slm_accuracy_fraction"}
missing = required_cols - set(dataset_df.columns)
if missing:
    raise KeyError(f"Missing required columns in {dataset_path}: {sorted(missing)}")

pareto_df = dataset_df[["id", "output_tokens_gpt5", "slm_accuracy_fraction"]].copy()
pareto_df = pareto_df.rename(columns={"id": "row_id", "output_tokens_gpt5": "output_tokens"})
pareto_df["output_tokens"] = pd.to_numeric(pareto_df["output_tokens"], errors="coerce")
pareto_df["slm_accuracy_fraction"] = pd.to_numeric(pareto_df["slm_accuracy_fraction"], errors="coerce")
pareto_df = pareto_df.dropna(subset=["output_tokens", "slm_accuracy_fraction"])
pareto_df = pareto_df[pareto_df["output_tokens"] >= 0]
pareto_df["complex"] = (pareto_df["slm_accuracy_fraction"] < 0.75).astype(int)

pareto_df = pareto_df.sort_values("output_tokens", ascending=False).reset_index(drop=True)
if pareto_df.empty:
    raise ValueError("No valid output-token values available for Pareto plot.")

# Cumulative token concentration
pareto_df["cum_tokens"] = pareto_df["output_tokens"].cumsum()
total_tokens = pareto_df["output_tokens"].sum()
pareto_df["cum_tokens_pct"] = (pareto_df["cum_tokens"] / total_tokens) * 100
pareto_df["row_pct"] = ((pareto_df.index + 1) / len(pareto_df)) * 100

# Cumulative composition of complex=1 within prefix rows
pareto_df["cum_complex_1"] = pareto_df["complex"].cumsum()
pareto_df["cum_complex_1_pct"] = (pareto_df["cum_complex_1"] / (pareto_df.index + 1)) * 100

fig, ax1 = plt.subplots(figsize=(9, 6))
ax1.plot(pareto_df["row_pct"], pareto_df["cum_tokens_pct"], linewidth=2, color="tab:blue", label="Cumulative token share")
ax1.axvline(20, linestyle="--", linewidth=1, color="gray", label="20% rows")
ax1.axhline(80, linestyle="--", linewidth=1, color="tab:red", label="80% tokens")
ax1.set_xlabel("% of rows sorted by output tokens (descending)")
ax1.set_ylabel("Cumulative % of total output tokens", color="tab:blue")
ax1.set_xlim(0, 100)
ax1.set_ylim(0, 100)
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(pareto_df["row_pct"], pareto_df["cum_complex_1_pct"], linestyle="--", linewidth=2, color="tab:orange", label="% complex = 1")
ax2.set_ylabel("% complex = 1", color="tab:orange")
ax2.set_ylim(0, 100)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower right")

plt.title("Cumulative GPT-5 Output Tokens + Amount of complex=1 by % of rows")
plt.tight_layout()
plt.show()

# Optional concentration readout
row_cutoff = int(np.ceil(0.2 * len(pareto_df)))
tokens_at_20pct_rows = pareto_df.iloc[row_cutoff - 1]["cum_tokens_pct"]
complex_at_20pct_rows = pareto_df.iloc[row_cutoff - 1]["cum_complex_1_pct"]
print(f"Top 20% of rows account for {tokens_at_20pct_rows:.2f}% of total output tokens.")
print(f"Within top 20% rows, complex=1 share is {complex_at_20pct_rows:.2f}%.")


In [ ]:
# Pareto-style cumulative distribution of output tokens + class composition
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.patches import Rectangle

dataset_path = "dataset/final_binary/final_dataset.csv"
dataset_df = pd.read_csv(dataset_path)

required_cols = {"id", "output_tokens_gpt5", "slm_accuracy_fraction"}
missing = required_cols - set(dataset_df.columns)
if missing:
    raise KeyError(f"Missing required columns in {dataset_path}: {sorted(missing)}")

pareto_df = dataset_df[["id", "output_tokens_gpt5", "slm_accuracy_fraction"]].copy()
pareto_df = pareto_df.rename(columns={"id": "row_id", "output_tokens_gpt5": "output_tokens"})
pareto_df["output_tokens"] = pd.to_numeric(pareto_df["output_tokens"], errors="coerce")
pareto_df["slm_accuracy_fraction"] = pd.to_numeric(pareto_df["slm_accuracy_fraction"], errors="coerce")
pareto_df = pareto_df.dropna(subset=["output_tokens", "slm_accuracy_fraction"])
pareto_df = pareto_df[pareto_df["output_tokens"] >= 0]
pareto_df["complex"] = (pareto_df["slm_accuracy_fraction"] < 0.75).astype(int)

pareto_df = pareto_df.sort_values("output_tokens", ascending=False).reset_index(drop=True)
if pareto_df.empty:
    raise ValueError("No valid output-token values available for Pareto plot.")

# Cumulative token concentration
pareto_df["cum_tokens"] = pareto_df["output_tokens"].cumsum()
total_tokens = pareto_df["output_tokens"].sum()
pareto_df["cum_tokens_pct"] = (pareto_df["cum_tokens"] / total_tokens) * 100
pareto_df["row_pct"] = ((pareto_df.index + 1) / len(pareto_df)) * 100

# Cumulative composition of complex=1 within prefix rows
pareto_df["cum_complex_1"] = pareto_df["complex"].cumsum()
pareto_df["cum_complex_1_pct"] = (pareto_df["cum_complex_1"] / (pareto_df.index + 1)) * 100

fig, ax1 = plt.subplots(figsize=(9, 6))
ax1.plot(pareto_df["row_pct"], pareto_df["cum_tokens_pct"], linewidth=2, color="tab:blue", label="Cumulative token share (%)")
ax1.plot(pareto_df["row_pct"], pareto_df["cum_complex_1_pct"], linestyle="--", linewidth=2, color="tab:orange", label="complex questions (%)")
ax1.axvline(20, linestyle="--", linewidth=1, color="gray", label="20% rows")
ax1.axhline(80, linestyle="--", linewidth=1, color="tab:red", label="80% tokens")
ax1.set_xlabel("% of rows sorted by output tokens (descending)")
ax1.set_ylabel("%")
ax1.set_xlim(0, 100)
ax1.set_ylim(0, 100)
ax1.grid(alpha=0.3)

lines1, labels1 = ax1.get_legend_handles_labels()
ax1.legend(lines1, labels1, loc="lower right")

plt.title("Cumulative GPT-5 Output Tokens + Amount of complex questions by % of rows")
plt.tight_layout()
plt.show()

# Optional concentration readout
row_cutoff = int(np.ceil(0.2 * len(pareto_df)))
tokens_at_20pct_rows = pareto_df.iloc[row_cutoff - 1]["cum_tokens_pct"]
complex_at_20pct_rows = pareto_df.iloc[row_cutoff - 1]["cum_complex_1_pct"]
print(f"Top 20% of rows account for {tokens_at_20pct_rows:.2f}% of total output tokens.")
print(f"Within top 20% rows, complex=1 share is {complex_at_20pct_rows:.2f}%.")


In [ ]:
# Confusion-matrix-style heatmap of model performance across domains
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

binary_dataset_path = "dataset/final_binary/final_dataset.csv"
qwen_regressor_dataset_path = "dataset/final_regressor/final_qwens_dataset.csv"
qwen_extra_models = ["qwen_0_5b", "qwen_1_5b", "qwen_3b", "qwen_14b"]


def model_accuracy_by_domain(dataset_path, selected_models=None):
    dataset_df = pd.read_csv(dataset_path, low_memory=False)
    model_cols = [col for col in dataset_df.columns if col.startswith("is_correct_")]

    if selected_models is not None:
        selected_model_cols = [f"is_correct_{model}" for model in selected_models]
        missing_selected = set(selected_model_cols) - set(dataset_df.columns)
        if missing_selected:
            raise KeyError(f"Missing selected model columns in {dataset_path}: {sorted(missing_selected)}")
        model_cols = selected_model_cols

    required_cols = {"dataset_name", *model_cols}
    missing = required_cols - set(dataset_df.columns)
    if missing:
        raise KeyError(f"Missing required columns in {dataset_path}: {sorted(missing)}")

    if not model_cols:
        raise ValueError(f"No model correctness columns found in {dataset_path}.")

    performance_df = dataset_df[["dataset_name", *model_cols]].copy()
    for col in model_cols:
        performance_df[col] = pd.to_numeric(performance_df[col], errors="coerce")

    domain_model_accuracy = performance_df.groupby("dataset_name")[model_cols].mean() * 100
    domain_model_accuracy = domain_model_accuracy.rename(
        columns={col: col.replace("is_correct_", "") for col in model_cols}
    )
    domain_model_accuracy = domain_model_accuracy.sort_index()

    total_accuracy = performance_df[model_cols].mean() * 100
    total_accuracy.index = [col.replace("is_correct_", "") for col in total_accuracy.index]

    model_domain_accuracy = domain_model_accuracy.T
    model_domain_accuracy["Total"] = total_accuracy
    return model_domain_accuracy


base_accuracy = model_accuracy_by_domain(binary_dataset_path)
qwen_extra_accuracy = model_accuracy_by_domain(
    qwen_regressor_dataset_path,
    selected_models=qwen_extra_models,
)

model_domain_accuracy = pd.concat([base_accuracy, qwen_extra_accuracy])
model_domain_accuracy = model_domain_accuracy.sort_values("Total", ascending=False)

fig, ax = plt.subplots(figsize=(9, 7.5))
heatmap = ax.imshow(model_domain_accuracy.values, cmap="YlGn", aspect="auto", vmin=0, vmax=100)

ax.set_xticks(range(len(model_domain_accuracy.columns)))
ax.set_xticklabels(model_domain_accuracy.columns, rotation=0)
ax.set_yticks(range(len(model_domain_accuracy.index)))
ax.set_yticklabels(model_domain_accuracy.index)
ax.set_xlabel("Domain")
ax.set_ylabel("Model")
ax.set_title("Model Performance Across Domains (%)")

for i in range(model_domain_accuracy.shape[0]):
    for j in range(model_domain_accuracy.shape[1]):
        value = model_domain_accuracy.iloc[i, j]
        ax.text(j, i, f"{value:.1f}", ha="center", va="center", color="black", fontsize=10)

total_col_idx = model_domain_accuracy.columns.get_loc("Total")
ax.axvline(total_col_idx - 0.5, color="black", linewidth=2)
ax.add_patch(
    Rectangle(
        (total_col_idx - 0.5, -0.5),
        1,
        model_domain_accuracy.shape[0],
        fill=False,
        edgecolor="black",
        linewidth=2.5,
    )
)

cbar = fig.colorbar(heatmap, ax=ax)
cbar.set_label("Accuracy (%)")

plt.tight_layout()
plt.show()

display(model_domain_accuracy.round(2))


In [ ]:
# Accuracy vs. average latency for the same combined model set
import pandas as pd
import matplotlib.pyplot as plt

binary_dataset_path = "dataset/final_binary/final_dataset.csv"
qwen_regressor_dataset_path = "dataset/final_regressor/final_qwens_dataset.csv"

model_sources = [
    (
        binary_dataset_path,
        [
            "gemma3_12b",
            "granite_3_2_8b",
            "llama3_1_8b",
            "qwen2_5_7b",
            "gpt4o_europe",
            "gpt5",
        ],
    ),
    (
        qwen_regressor_dataset_path,
        [
            "qwen_0_5b",
            "qwen_1_5b",
            "qwen_3b",
            "qwen_14b",
        ],
    ),
]

accuracy_latency_rows = []
for dataset_path, models in model_sources:
    dataset_df = pd.read_csv(dataset_path, low_memory=False)

    for model in models:
        accuracy_col = f"is_correct_{model}"
        latency_col = f"latency_{model}"
        missing = {accuracy_col, latency_col} - set(dataset_df.columns)
        if missing:
            raise KeyError(f"Missing columns in {dataset_path}: {sorted(missing)}")

        accuracy = pd.to_numeric(dataset_df[accuracy_col], errors="coerce").mean() * 100
        avg_latency = pd.to_numeric(dataset_df[latency_col], errors="coerce").mean()
        accuracy_latency_rows.append(
            {
                "model": model,
                "total_accuracy": accuracy,
                "avg_latency": avg_latency,
            }
        )

accuracy_latency_df = pd.DataFrame(accuracy_latency_rows).sort_values("total_accuracy", ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    accuracy_latency_df["avg_latency"],
    accuracy_latency_df["total_accuracy"],
    s=90,
    color="tab:blue",
    alpha=0.85,
)

label_settings = {
    "gpt5": {"offset": (-12, 10), "ha": "right", "va": "bottom"},
    "qwen_14b": {"offset": (12, 13), "ha": "left", "va": "bottom"},
    "qwenmax_12b": {"offset": (12, -12), "ha": "left", "va": "top"},
    "llama3_1_8b": {"offset": (12, 12), "ha": "left", "va": "bottom"},
    "qwen2_5_7b": {"offset": (12, -10), "ha": "left", "va": "top"},
    "qwen_3b": {"offset": (12, 10), "ha": "left", "va": "bottom"},
    "granite_3_2_8b": {"offset": (12, -10), "ha": "left", "va": "top"},
}

for _, row in accuracy_latency_df.iterrows():
    settings = label_settings.get(row["model"], {"offset": (12, 10), "ha": "left", "va": "bottom"})
    ax.annotate(
        row["model"],
        (row["avg_latency"], row["total_accuracy"]),
        xytext=settings["offset"],
        textcoords="offset points",
        ha=settings["ha"],
        va=settings["va"],
        fontsize=9,
        arrowprops={"arrowstyle": "-", "color": "0.45", "lw": 0.7, "shrinkA": 4, "shrinkB": 6},
        bbox={"boxstyle": "round,pad=0.15", "fc": "white", "ec": "none", "alpha": 0.75},
    )

ax.set_xlabel("Average latency (seconds)")
ax.set_ylabel("Total accuracy (%)")
ax.set_title("Model Accuracy vs. Average Latency")
ax.margins(x=0.08, y=0.08)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

display(accuracy_latency_df.round(2))


In [ ]:
# Print a summary of number of complex = 1 and complex = 0
complex_counts = pareto_df["complex"].value_counts().sort_index()
print("Complexity Class Distribution:")
print(f"Complex = 0: {complex_counts.get(0, 0)} rows")
print(f"Complex = 1: {complex_counts.get(1, 0)} rows")
print(f"% of complex = 0: {(complex_counts.get(0, 0) / len(pareto_df) * 100):.2f}%")
# Print a summary of the number of accuracy fraction in the other dataset
accuracy_fraction_counts = dataset_df["slm_accuracy_fraction"].value_counts().sort_index()
print("\nSLM Accuracy Fraction Distribution:")
for fraction, count in accuracy_fraction_counts.items():
    print(f"SLM Accuracy Fraction = {fraction}: {count} rows")

In [ ]:
# Histogram of row counts for each SLM accuracy fraction
slm_accuracy = pd.to_numeric(dataset_df["slm_accuracy_fraction"], errors="coerce").dropna()
accuracy_fraction_summary = slm_accuracy.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(
    accuracy_fraction_summary.index,
    accuracy_fraction_summary.values,
    width=0.12,
    color="tab:blue",
    alpha=0.75,
    edgecolor="black",
    linewidth=0.6,
)
ax.set_xlim(-0.15, 1.15)
ax.set_xticks(accuracy_fraction_summary.index)
ax.set_xticklabels([f"{fraction:.1f}" for fraction in accuracy_fraction_summary.index])
ax.set_xlabel("SLM Accuracy Fraction")
ax.set_ylabel("Number of rows")
ax.set_title("Distribution of SLM Accuracy Fraction")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots for GPT-5 token counts and latency
import pandas as pd
import matplotlib.pyplot as plt

scatter_dataset_path = "dataset/final_binary/final_dataset.csv"
scatter_df = pd.read_csv(scatter_dataset_path, low_memory=False)

scatter_cols = {
    "input_tokens": "input_tokens_gpt5",
    "output_tokens": "output_tokens_gpt5",
    "latency": "latency_gpt5",
}
missing = set(scatter_cols.values()) - set(scatter_df.columns)
if missing:
    raise KeyError(f"Missing required columns in {scatter_dataset_path}: {sorted(missing)}")

scatter_plot_df = scatter_df[list(scatter_cols.values())].rename(
    columns={column: label for label, column in scatter_cols.items()}
)
for column in scatter_plot_df.columns:
    scatter_plot_df[column] = pd.to_numeric(scatter_plot_df[column], errors="coerce")
scatter_plot_df = scatter_plot_df.dropna()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
scatter_style = {"alpha": 0.35, "s": 14, "edgecolors": "none"}

axes[0].scatter(scatter_plot_df["input_tokens"], scatter_plot_df["output_tokens"], **scatter_style)
axes[0].set_xlabel("Input tokens")
axes[0].set_ylabel("Output tokens")
axes[0].set_title("Output Tokens vs. Input Tokens")

axes[1].scatter(scatter_plot_df["input_tokens"], scatter_plot_df["latency"], **scatter_style)
axes[1].set_xlabel("Input tokens")
axes[1].set_ylabel("Latency")
axes[1].set_title("Latency vs. Input Tokens")

axes[2].scatter(scatter_plot_df["latency"], scatter_plot_df["output_tokens"], **scatter_style)
axes[2].set_xlabel("Latency")
axes[2].set_ylabel("Output tokens")
axes[2].set_title("Output Tokens vs. Latency")

for ax in axes:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()